In [15]:

import sys

from sqlalchemy import create_engine
from sqlalchemy import text, bindparam
import pandas as pd

from nhs_waiting_lists.constants import proj_db_path, wait_ranges_lt_18
from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()


In [16]:

DB_PATH = project_root / proj_db_path / "nhs_rttwtd.db"
DATA_DIR = "./data"

conn = create_engine(f"sqlite:///{DB_PATH}")

In [17]:
PROVIDER_CODES = ['R0B', 'RAJ', 'RTH', 'RTE', "RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW",
                  "RDU", "RH8", "RWY", "RXC", "RL4", "RDE", "RXK", "RXR", "RJ2", "RN5", "RHU",
                  "RGN", "RWP", "RWD", "RAJ"]
TREATMENT_CODES = ('C_101', 'C_110', 'C_320', 'C_330', 'C_400', 'C_502', 'C_301', 'C_999')

In [87]:
query = text("""
             SELECT p.provider_name as Provider,
                    (SELECT c1_wait_pct_lt_18
                     FROM (SELECT c1.period c1_period,
                                  'Q' ||
                                  COALESCE(NULLIF((strftime('%m', c1.period || '-01') + 2) / 3, 0), 4) AS c1_Quarter,
                                  substr(strftime('%Y', c1.period || '-01'), 3, 2)                     AS dt_yy,
                                  p1.provider_name,
                                  AVG(c1.wait_pct_lt_18) c1_wait_pct_lt_18,
                                  p1.provider_name
                           FROM consolidated AS c1
                                    INNER JOIN providers AS p1 ON c1.provider = p1.provider_code
                           WHERE p.provider_code = p1.provider_code
                           GROUP BY dt_yy, c1_Quarter
                           ORDER BY dt_yy DESC, c1_Quarter DESC)
                     LIMIT 1)       as q_1
             FROM providers p
             WHERE p.provider_code IN :provider_codes
             ORDER BY provider_name; \
             """).bindparams(
    bindparam('provider_codes', expanding=True),
    # bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(query, conn, params={ # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES,
}
                 )
# df.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")
df

,Provider,q_1
0,Calderdale and Huddersfield NHS Foundation Trust,0.729257
1,East Lancashire Hospitals NHS Trust,0.679058
2,East Suffolk and North Essex NHS Foundation Trust,0.671413
3,East Sussex Healthcare NHS Trust,0.683843
4,East and North Hertfordshire NHS Trust,0.674998
5,Frimley Health NHS Foundation Trust,0.605131
6,Gloucestershire Hospitals NHS Foundation Trust,0.725736
7,Hampshire Hospitals NHS Foundation Trust,0.607850
8,Lewisham and Greenwich NHS Trust,0.680004
9,Maidstone and Tunbridge Wells NHS Trust,0.779117


In [25]:
df.query("provider == 'RAJ' and treatment == 'C_999'")

,period,provider,treatment,incomplete,admitted,nonadmitted,new_periods,incomplete_prev,wait_pct_lt_18,provider_name
1925,2021-05,RAJ,C_999,97931,3434,19825,32552,NaN,0.685912,Mid and South Essex NHS Foundation Trust
1926,2021-06,RAJ,C_999,100319,3980,22894,30818,97931.0,0.693906,Mid and South Essex NHS Foundation Trust
1927,2021-07,RAJ,C_999,107770,3643,21980,34581,100319.0,0.698784,Mid and South Essex NHS Foundation Trust
1928,2021-08,RAJ,C_999,113630,3321,20095,32564,107770.0,0.680049,Mid and South Essex NHS Foundation Trust
1929,2021-09,RAJ,C_999,116067,3481,21776,34859,113630.0,0.657973,Mid and South Essex NHS Foundation Trust
1930,2021-10,RAJ,C_999,119249,3429,22964,35104,116067.0,0.652978,Mid and South Essex NHS Foundation Trust
1931,2021-11,RAJ,C_999,119629,3647,24487,41772,119249.0,0.638558,Mid and South Essex NHS Foundation Trust
1932,2021-12,RAJ,C_999,121498,2656,20327,37036,119629.0,0.611022,Mid and South Essex NHS Foundation Trust
1933,2022-01,RAJ,C_999,124212,2696,20741,39003,121498.0,0.603034,Mid and South Essex NHS Foundation Trust
1934,2022-02,RAJ,C_999,131003,3055,21162,37249,124212.0,0.598444,Mid and South Essex NHS Foundation Trust


In [90]:
df = pd.read_sql("""
SELECT * from v_consolidated
""", conn, params={ # type: ignore[arg-type]
    'provider_codes': PROVIDER_CODES,
}
                 )
df

,period,provider_name,subtype,Quarter,provider,treatment,incomplete,admitted,nonadmitted,new_periods,incomplete_prev,wait_pct_lt_18
0,2021-05,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,38116,2469,10814,17215,NaN,0.875669
1,2021-06,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,40317,2772,11371,18622,38116.0,0.881812
2,2021-07,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43170,2420,10116,17791,40317.0,0.869099
3,2021-08,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43946,2356,8898,15761,43170.0,0.856528
4,2021-09,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,45349,2790,10123,18007,43946.0,0.844848
...,...,...,...,...,...,...,...,...,...,...,...,...
1186,2025-04,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,60237,1755,7215,11017,60809.0,0.564620
1187,2025-05,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,57660,2012,7897,11771,60237.0,0.584287
1188,2025-06,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,58192,2172,8484,13750,57660.0,0.606853
1189,2025-07,East Lancashire Hospitals NHS Trust,Acute - Large,3,RXR,C_999,57371,2368,8695,14741,58192.0,0.611685


In [92]:
# extract year + quarter
df["year"] = df["period"].str[:4].astype(int)
df["month"] = df["period"].str[5:7].astype(int)
df["quarter"] = ((df["month"] - 1) // 3) + 1


# average by provider/year/quarter
q = (
    df.groupby(["provider", "year", "quarter"])["wait_pct_lt_18"]
    .mean()
    .reset_index()
)

# focus on 2024
q_2024 = q[q["year"] == 2024]

# pivot to quarters as columns
pivot = q_2024.pivot(index="provider", columns="quarter", values="wait_pct_lt_18")
pivot.columns = [f"2024-Q{c}" for c in pivot.columns]

# compute trend vs 3 quarters earlier (or 2 here)
pivot["trend_delta"] = pivot["2024-Q4"] - pivot["2024-Q2"]
pivot["Trend"] = pd.cut(
    pivot["trend_delta"],
    bins=[-1, -0.01, 0.01, 1],
    labels=["↓ Declining", "→ Stable", "↑ Improving"]
)

# format as percent
for col in ["2024-Q2", "2024-Q3", "2024-Q4"]:
    pivot[col] = (pivot[col] * 100).round(1).astype(str) + "%"

pivot = pivot.reset_index()
print(pivot)



   provider   2024-Q1 2024-Q2 2024-Q3 2024-Q4  trend_delta        Trend
0       R0B  0.723882   72.8%   73.1%   74.6%     0.018008  ↑ Improving
1       RAJ  0.519307   52.9%   52.3%   52.2%    -0.007135     → Stable
2       RDE  0.567807   57.5%   56.0%   54.9%    -0.025253  ↓ Declining
3       RDU  0.482678   49.3%   49.9%   51.0%     0.017343  ↑ Improving
4       REF  0.637670   67.0%   67.8%   68.6%     0.015699  ↑ Improving
5       RGN  0.490229   50.1%   51.7%   52.4%     0.023778  ↑ Improving
6       RH8  0.546463   56.5%   57.2%   58.0%     0.015716  ↑ Improving
7       RHU  0.530068   55.0%   54.0%   54.2%    -0.007404     → Stable
8       RHW  0.832514   83.4%   82.3%   81.4%    -0.019590  ↓ Declining
9       RJ2  0.542139   56.1%   55.7%   56.6%     0.004180     → Stable
10      RL4  0.527360   52.6%   53.7%   52.8%     0.001571     → Stable
11      RN5  0.566778   57.5%   57.2%   58.0%     0.004776     → Stable
12      RTE  0.656900   65.9%   65.3%   66.3%     0.004445     →